# ⛰️ Mountain Entity Detection Demo
This notebook demonstrates the inference capabilities of a custom Named Entity Recognition (NER) model. 

**Project Overview:**
* **Objective:** to automatically detect and extract mountain names (the `MOUNTAIN` entity) from unstructured text.
* **Model Architecture:** the model is built on top of the `microsoft/deberta-v3-base` architecture and fine-tuned for token classification using TensorFlow/Keras.
* **Functionality:** this demo utilizes the custom `MountainDetector` class to process input text, handle subword tokenization, and provide visual HTML-based highlighting of the detected entities.

In [1]:
# !pip install -r requirements.txt

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent)) 

In [3]:
from src.model_inference import MountainDetector

MODEL_PATH = "../models/deberta_ner_model" 
detector = MountainDetector(MODEL_PATH)

The tokenizer you are loading from '../models/deberta_ner_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFDebertaV2ForTokenClassification.

All the layers of TFDebertaV2ForTokenClassification were initialized from the model checkpoint at ../models/deberta_ner_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDebertaV2ForTokenClassification for predictions without further training.


In [4]:
from src.model_training import evaluate_model
evaluate_model("../data/processed/test.json", detector.model, detector.tokenizer)

Loading test data from ../data/processed/test.json...


Map:   0%|          | 0/214 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/Users/aleksandralitvak/Documents/codes/python/Quantum Test Tasks/Natural Language Processing. Named entity recognition/venv_ner/lib/python3.13/site-packages/datasets/arrow_dataset.py:419: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(



--- Evaluation Summary ---
Scenario: all

              correct   incorrect     partial      missed    spurious   precision      recall    f1-score

ent_type          170           0           0           1           0        1.00        0.99        1.00
   exact          170           0           0           1           0        1.00        0.99        1.00
 partial          170           0           0           1           0        1.00        0.99        1.00
  strict          170           0           0           1           0        1.00        0.99        1.00


--- Error аnalysis ---
Sentence: Returning from Kangchenjunga , the researcher faced a mountain of unread emails .
 -> [MISSED (False Negative)] Word: 'Kangchenjunga' | Reality: B-MOUNTAIN | Prediction: O
------------------------------------------------------------


In [5]:
demo_test_data = [
    # Basic cases: memorization vs. сontext (Real vs. Fictional)
    {
        ## real mountain (seen in training)
        "tokens": ["My", "friends", "and", "I", "are", "climbing", "to", "Everest", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "O"]
    },
    {
        ## fictional mountain (zero-shot generalization)
        "tokens": ["My", "friends", "and", "I", "are", "climbing", "to", "Mount", "Zorblax", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "I-MOUNTAIN", "O"]
    },
    
    # Entity overlap (City vs. Mountain)
    {
        ## real overlap (city and mountain share the same name)
        "tokens": ["After", "visiting", "Washington", "D.C.", ",", "we", "decided", "to", "hike", "Mount", "Washington", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "I-MOUNTAIN", "O"]
    },
    {
        ## fictional overlap (city and mountain share the same name)
        "tokens": ["After", "visiting", "the", "city", "of", "Eldoriapok", ",", "we", "decided", "to", "hike", "Mount", "Eldoriapok", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "I-MOUNTAIN", "O"]
    },
    {
        ## fictional overlap without Mount (city and mountain share the same name)
        "tokens": ["After", "visiting", "the", "city", "of", "Eldoria", ",", "we", "decided", "to", "hike", "Eldoriapok", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "O"]
    },
    
    # Parts of speech (Verbs & Metaphors vs. Geography)
    {
        ## real mountain with metaphor ("mount" as verb, "mountain" as metaphor)
        "tokens": ["To", "mount", "a", "winter", "expedition", "to", "K2", ",", "you", "must", "overcome", "a", "mountain", "of", "challenges", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "O", "O", "O", "O", "O", "O", "O", "O", "O"]
    },
    {
        ## real mountain with metaphor ("peak" as metaphor vs Pikes Peak)
        "tokens": ["She", "reached", "the", "peak", "of", "her", "career", "shortly", "before", "climbing", "Pikes", "Peak", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "I-MOUNTAIN", "O"]
    },
    {
        ## fictional mountain with metaphor
        "tokens": ["They", "overcame", "a", "mountain", "of", "paperwork", "just", "to", "get", "a", "permit", "for", "Zirkon", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "O"]
    },
    
    # Brands in geographical context
    {
        ## Ford Everest (brand) vs Mount Elbrus (real mountain)
        "tokens": ["We", "drove", "our", "new", "Ford", "Everest", "to", "the", "base", "of", "Mount", "Elbrus", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-MOUNTAIN", "I-MOUNTAIN", "O"]
    },
    
    # Hard negatives & known limitations 
    {
        ## completely different geographical entity (City/Country)
        "tokens": ["Kyiv", "is", "the", "beautiful", "capital", "of", "Ukraine", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O"]
    },
    {
        ## deceptive city name containing "Mountain"
        "tokens": ["He", "lives", "in", "Mountain", "View", ",", "California", ",", "near", "the", "headquarters", "of", "Google", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]
    },
    {
        ## other topography (valley, rockfall, summit as common noun)
        "tokens": ["The", "massive", "rockfall", "near", "the", "Callejón", "de", "Huaylas", "valley", "blocked", "the", "path", "to", "the", "summit", "."],
        "ner_tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]
    }
]

evaluate_model(demo_test_data, detector.model, detector.tokenizer, batch_size=16)

Map:   0%|          | 0/12 [00:00<?, ? examples/s]


--- Evaluation Summary ---
Scenario: all

              correct   incorrect     partial      missed    spurious   precision      recall    f1-score

ent_type            9           0           0           0           0        1.00        1.00        1.00
   exact            9           0           0           0           0        1.00        1.00        1.00
 partial            9           0           0           0           0        1.00        1.00        1.00
  strict            9           0           0           0           0        1.00        1.00        1.00


--- Error аnalysis ---


## Conclusion

Overall, the fine-tuned DeBERTa model showed great results in identifying mountain names. These scores are reliable because the dataset was carefully split to avoid data leakage. Since no mountain name was repeated across the train and test sets, it is clear that the model learned the language context, rather than just memorizing words.

To further increase confidence in the model, the next step should be to make the dataset bigger. Just like in statistics, where a larger sample size gives more reliable results (a lower p-value), more data will help prove the model's stability. A bigger dataset will cover a wider variety of sentence structures and rare cases.

Despite the high scores, the model is not perfect yet. During the error analysis, exactly one mountain was missed (False Negative): 
> *"Returning from Kangchenjunga, the researcher faced a mountain of unread emails."*

The model missed the word "Kangchenjunga" because there were only two words of context before it ("Returning from"). This was not enough information. At the same time, the second part of the sentence had a very strong distractive metaphor ("faced a mountain of..."). Because the real geographical context was too short, the model got confused by the metaphor and missed the actual mountain.